# DL Mini-Project: Digits Classifier in NumPy

Build a 2-layer MLP to classify handwritten digits (sklearn digits dataset) using only NumPy.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits


## Step 1: Load and prepare data

In [ ]:
digits = load_digits()
X_all = digits.data / 16.0
y_all = digits.target
print(f'Full dataset: {X_all.shape[0]} samples, {X_all.shape[1]} features, {len(np.unique(y_all))} classes')

# Use all 1797 samples for better accuracy
def one_hot(y, n_classes=10):
    Y = np.zeros((len(y), n_classes))
    Y[np.arange(len(y)), y] = 1
    return Y

Y_all = one_hot(y_all)
np.random.seed(42)
idx = np.random.permutation(len(X_all))
split = int(0.8 * len(idx))
X_train, X_test = X_all[idx[:split]], X_all[idx[split:]]
Y_train, Y_test = Y_all[idx[:split]], Y_all[idx[split:]]
y_test = y_all[idx[split:]]
print(f'Train: {len(X_train)}, Test: {len(X_test)}')


## Step 2: Initialize MLP (64 → 128 → 64 → 10)

In [ ]:
np.random.seed(42)
W1 = np.random.randn(128, 64) * 0.01; b1 = np.zeros(128)
W2 = np.random.randn(64, 128) * 0.01; b2 = np.zeros(64)
W3 = np.random.randn(10, 64) * 0.01; b3 = np.zeros(10)
total = W1.size + b1.size + W2.size + b2.size + W3.size + b3.size
print(f'Total parameters: {total:,}')


## Step 3: Training loop

In [ ]:
relu = lambda z: np.maximum(0, z)
def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

lr = 0.1; epochs = 300; batch_size = 32; losses = []

for epoch in range(epochs):
    p = np.random.permutation(len(X_train))
    Xb, Yb = X_train[p], Y_train[p]
    epoch_loss = 0
    for i in range(0, len(Xb), batch_size):
        xb = Xb[i:i+batch_size]; yb = Yb[i:i+batch_size]
        z1 = xb @ W1.T + b1; a1 = relu(z1)
        z2 = a1 @ W2.T + b2; a2 = relu(z2)
        z3 = a2 @ W3.T + b3; a3 = softmax(z3)
        loss = -np.mean(np.sum(yb * np.log(a3 + 1e-9), axis=1))
        epoch_loss += loss
        d3 = (a3 - yb) / len(xb)
        dW3 = d3.T @ a2; db3 = d3.sum(0)
        d2 = (d3 @ W3) * (z2 > 0); dW2 = d2.T @ a1; db2 = d2.sum(0)
        d1 = (d2 @ W2) * (z1 > 0); dW1 = d1.T @ xb; db1 = d1.sum(0)
        W3 -= lr*dW3; b3 -= lr*db3
        W2 -= lr*dW2; b2 -= lr*db2
        W1 -= lr*dW1; b1 -= lr*db1
    losses.append(epoch_loss)
    if (epoch+1) % 60 == 0:
        print(f'Epoch {epoch+1}: loss = {epoch_loss:.4f}')


## Step 4: Plot loss curve

In [ ]:
plt.plot(losses, linewidth=1.5, color='steelblue')
plt.xlabel('Epoch'); plt.ylabel('Cross-entropy loss'); plt.title('Training loss')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f'Final loss: {losses[-1]:.4f}')


## Step 5: Evaluate on test set

In [ ]:
z1t = X_test @ W1.T + b1; a1t = relu(z1t)
z2t = a1t @ W2.T + b2; a2t = relu(z2t)
z3t = a2t @ W3.T + b3; a3t = softmax(z3t)
preds = a3t.argmax(axis=1)
acc = np.mean(preds == y_test)
print(f'Test accuracy: {acc*100:.1f}%')

# Confusion summary
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(8,6))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix')
plt.tight_layout(); plt.show()
